## 1 .Importing Libraries


In [5]:
import os

import numpy as np

import pandas as pd

import sys

from sklearn.model_selection import train_test_split

## 2.Reading Data

In [6]:
Project_Dir=r'/Users/akashjaiswal/Desktop/amex-hackathon-2'
Data_dir="Data"

In [7]:
def get_data(name):
    file_name = f"{name}.parquet"
    file_path = os.path.join(Project_Dir, Data_dir, file_name)
    return pd.read_parquet(file_path, engine="pyarrow")

# Load train and validation sets
train = get_data("train_data")
# val = get_data("train_data_val")
test = get_data("test_data")

print(f"Train: {len(train)} rows")
# print(f"Val: {len(val)} rows")
print(f"Test: {len(test)} rows")

Train: 770164 rows
Test: 369301 rows


In [8]:
# ── Dataset size comparison: small vs full ────────────────────────────────────
small_train = get_data("small_train")
small_test  = get_data("small_test")
full_train  = get_data("full_train")
full_test   = get_data("full_test")

datasets = {
    "small_train": small_train,
    "small_test":  small_test,
    "full_train":  full_train,
    "full_test":   full_test,
}

summary = pd.DataFrame({
    name: {
        "rows":        df.shape[0],
        "columns":     df.shape[1],
        "memory (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "null %":      round(df.isnull().mean().mean() * 100, 2),
        "dtypes":      str(df.dtypes.value_counts().to_dict()),
    }
    for name, df in datasets.items()
}).T

print(summary.to_string())

# ── Column consistency check ───────────────────────────────────────────────────
small_cols = set(small_train.columns)
full_cols  = set(full_train.columns)

only_in_small = small_cols - full_cols
only_in_full  = full_cols  - small_cols

print(f"\nColumns only in small_train : {only_in_small or 'None'}")
print(f"Columns only in full_train  : {only_in_full  or 'None'}")
print(f"Columns match               : {small_cols == full_cols}")

               rows columns memory (MB) null %                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770164 entries, 0 to 770163
Columns: 372 entries, id1 to f366
dtypes: object(372)
memory usage: 2.1+ GB


In [10]:
train['id1']

0         1366776_189706075_16-23_2023-11-02 22:22:00.042
1             1366776_89227_16-23_2023-11-01 23:51:24.999
2             1366776_35046_16-23_2023-11-01 00:30:59.797
3           1366776_6275451_16-23_2023-11-02 22:21:32.261
4             1366776_78053_16-23_2023-11-02 22:21:34.799
                               ...                       
770159        1896641_87731_16-23_2023-11-02 08:14:21.524
770160       1896641_505604_16-23_2023-11-02 08:14:24.458
770161        1896641_25212_16-23_2023-11-02 08:14:25.748
770162        1900765_95157_16-23_2023-11-02 06:08:25.900
770163        1901215_95807_16-23_2023-11-01 11:01:33.086
Name: id1, Length: 770164, dtype: object

In [11]:
train

,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,28.0,0.0,0.0,337.0,0.0,0.0
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,87.0,0.0,0.0,1010.0,2.0,0.0019801980198019
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,23.0,0.0,0.0,1010.0,2.0,0.0019801980198019
3,1366776_6275451_16-23_2023-11-02 22:21:32.261,1366776,6275451,2023-11-02 22:21:32.261,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,277.0,1.0,0.003610108303249,337.0,0.0,0.0
4,1366776_78053_16-23_2023-11-02 22:21:34.799,1366776,78053,2023-11-02 22:21:34.799,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,359.0,0.0,0.0,337.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
770159,1896641_87731_16-23_2023-11-02 08:14:21.524,1896641,87731,2023-11-02 08:14:21.524,2023-11-02,0,None,None,None,None,...,0.0021005417064525,0.0404038812929242,0.0,None,90.0,1.0,0.0111111111111111,282.0,1.0,0.0035460992907801
770160,1896641_505604_16-23_2023-11-02 08:14:24.458,1896641,505604,2023-11-02 08:14:24.458,2023-11-02,0,None,None,None,None,...,-0.0005843979733278,0.0506497784165768,0.0,None,33.0,0.0,0.0,58.0,1.0,0.0172413793103448
770161,1896641_25212_16-23_2023-11-02 08:14:25.748,1896641,25212,2023-11-02 08:14:25.748,2023-11-02,0,None,None,None,None,...,0.0003542232402053,0.0498707994939233,0.0,None,33.0,0.0,0.0,58.0,1.0,0.0172413793103448
770162,1900765_95157_16-23_2023-11-02 06:08:25.900,1900765,95157,2023-11-02 06:08:25.900,2023-11-02,0,None,None,None,None,...,None,None,0.0,None,None,None,None,None,None,None


In [12]:
train.dtypes

id1     object
id2     object
id3     object
id4     object
id5     object
         ...  
f362    object
f363    object
f364    object
f365    object
f366    object
Length: 372, dtype: object

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770164 entries, 0 to 770163
Columns: 372 entries, id1 to f366
dtypes: object(372)
memory usage: 2.1+ GB


## converted dtype of event_time_stamp and event_date cols from object to to_datetime

In [14]:
for col in ['id4', 'id5']:
    train[col] = pd.to_datetime(train[col])

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770164 entries, 0 to 770163
Columns: 372 entries, id1 to f366
dtypes: datetime64[ns](2), object(370)
memory usage: 2.1+ GB


In [16]:
object_cols=train.select_dtypes(include=['object']).columns
print("object cols:")
print(object_cols)

object cols:
Index(['id1', 'id2', 'id3', 'y', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6',
       ...
       'f357', 'f358', 'f359', 'f360', 'f361', 'f362', 'f363', 'f364', 'f365',
       'f366'],
      dtype='object', length=370)


In [17]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770164 entries, 0 to 770163
Columns: 372 entries, id1 to f366
dtypes: datetime64[ns](2), object(370)
memory usage: 2.1+ GB


In [18]:
data_dict_path = "/Users/akashjaiswal/Desktop/amex_hackathon/Data/data_dictionary.csv"
data_dict = pd.read_csv(data_dict_path)

## dropped rows of ohe in which missing values were there,these were only 18 rows

In [19]:
# Get all OHE columns
ohe_cols = data_dict[data_dict['Type'] == 'One hot encoded']['masked_column'].tolist()

print(f"Found {len(ohe_cols)} OHE columns")
print(f"Rows before: {len(train)}")

# Drop rows where ANY OHE column is null
train = train.dropna(subset=ohe_cols).copy()

print(f"Rows after: {len(train)}")
print(f"Dropped {len(train) - len(train)} rows")

Found 84 OHE columns
Rows before: 770164
Rows after: 770146
Dropped 0 rows


In [20]:
train.dtypes

id1             object
id2             object
id3             object
id4     datetime64[ns]
id5     datetime64[ns]
             ...      
f362            object
f363            object
f364            object
f365            object
f366            object
Length: 372, dtype: object

## converted dtype from object to 'uint8'

In [21]:
for col in ohe_cols:
    # pd.to_numeric(df[col], errors='coerce').astype('uint8')
    train[col]=pd.to_numeric(train[col], errors='coerce').astype('uint8')

## Handling Null Values

In [22]:
# print a table showing null values in each column percentage and sort in decreasing order like a table
null_values = train.isnull().sum() / len(train) * 100
null_values = null_values.sort_values(ascending=False)
print(null_values)

f135    100.000000
f122    100.000000
f136    100.000000
f112    100.000000
f80      99.989872
           ...    
f270      0.000000
f271      0.000000
f272      0.000000
f273      0.000000
id1       0.000000
Length: 372, dtype: float64


In [23]:
# show the columns with more than 70 percent null values in train
train.columns[train.isnull().mean() > 0.7]

Index(['f3', 'f4', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20',
       'f21', 'f29', 'f33', 'f34', 'f35', 'f36', 'f37', 'f40', 'f64', 'f66',
       'f70', 'f79', 'f80', 'f81', 'f84', 'f88', 'f92', 'f112', 'f114', 'f116',
       'f117', 'f118', 'f119', 'f120', 'f121', 'f122', 'f135', 'f136', 'f154',
       'f176', 'f187', 'f188', 'f189', 'f190', 'f191', 'f192', 'f193', 'f194',
       'f195', 'f196', 'f197', 'f205', 'f218', 'f220', 'f221', 'f360'],
      dtype='object')

## dropped all cols in which missing values were>70%

In [24]:
cols_to_drop = train.columns[train.isnull().mean() > 0.7]
train = train.drop(cols_to_drop, axis=1)

In [25]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 770146 entries, 0 to 770163
Columns: 316 entries, id1 to f366
dtypes: datetime64[ns](2), object(230), uint8(84)
memory usage: 1.4+ GB


## converted all numeric cols to float32

In [26]:
# now for all the columns in data_dict with type Numerical convert those to float in train, val 
for col in data_dict[data_dict['Type']=='Numerical']['masked_column']:
    if(col=='id4' or col=='id5'):
        continue
    if col not in train.columns:
        # print(f"Column {col} is missing in one of the datasets.")
        continue
    train[col] = train[col].astype('float32')

In [27]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 770146 entries, 0 to 770163
Columns: 316 entries, id1 to f366
dtypes: datetime64[ns](2), float32(215), object(15), uint8(84)
memory usage: 799.1+ MB


In [28]:
for col in data_dict[data_dict['Type']=='Categorical']['masked_column']:
    train[col] = train[col].astype('category')
    

In [29]:
for col in data_dict[data_dict['Type']=='Key']['masked_column']:
    train[col] = train[col].astype('category')

In [30]:
for col in data_dict[data_dict['Type']=='Label']['masked_column']:
    train[col] = pd.to_numeric(train[col], errors='coerce').astype('int8')

In [31]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 770146 entries, 0 to 770163
Columns: 316 entries, id1 to f366
dtypes: category(13), datetime64[ns](2), float32(215), int8(1), object(1), uint8(84)
memory usage: 752.1+ MB


In [32]:
train

,id1,id2,id3,id4,id5,y,f1,f2,f5,f6,...,f356,f357,f358,f359,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,28.0,0.0,0.000000,337.0,0.0,0.000000
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,87.0,0.0,0.000000,1010.0,2.0,0.001980
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,23.0,0.0,0.000000,1010.0,2.0,0.001980
3,1366776_6275451_16-23_2023-11-02 22:21:32.261,1366776,6275451,2023-11-02 22:21:32.261,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,277.0,1.0,0.003610,337.0,0.0,0.000000
4,1366776_78053_16-23_2023-11-02 22:21:34.799,1366776,78053,2023-11-02 22:21:34.799,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,359.0,0.0,0.000000,337.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
770159,1896641_87731_16-23_2023-11-02 08:14:21.524,1896641,87731,2023-11-02 08:14:21.524,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.003209,0.002101,0.040404,0.0,90.0,1.0,0.011111,282.0,1.0,0.003546
770160,1896641_505604_16-23_2023-11-02 08:14:24.458,1896641,505604,2023-11-02 08:14:24.458,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.001635,-0.000584,0.050650,0.0,33.0,0.0,0.000000,58.0,1.0,0.017241
770161,1896641_25212_16-23_2023-11-02 08:14:25.748,1896641,25212,2023-11-02 08:14:25.748,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.002612,0.000354,0.049871,0.0,33.0,0.0,0.000000,58.0,1.0,0.017241
770162,1900765_95157_16-23_2023-11-02 06:08:25.900,1900765,95157,2023-11-02 06:08:25.900,2023-11-02,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
train_copy=train.copy()

In [34]:
train_copy

,id1,id2,id3,id4,id5,y,f1,f2,f5,f6,...,f356,f357,f358,f359,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,28.0,0.0,0.000000,337.0,0.0,0.000000
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,87.0,0.0,0.000000,1010.0,2.0,0.001980
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,23.0,0.0,0.000000,1010.0,2.0,0.001980
3,1366776_6275451_16-23_2023-11-02 22:21:32.261,1366776,6275451,2023-11-02 22:21:32.261,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,277.0,1.0,0.003610,337.0,0.0,0.000000
4,1366776_78053_16-23_2023-11-02 22:21:34.799,1366776,78053,2023-11-02 22:21:34.799,2023-11-02,0,1.0,NaN,NaN,NaN,...,NaN,NaN,-9999.000000,0.0,359.0,0.0,0.000000,337.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
770159,1896641_87731_16-23_2023-11-02 08:14:21.524,1896641,87731,2023-11-02 08:14:21.524,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.003209,0.002101,0.040404,0.0,90.0,1.0,0.011111,282.0,1.0,0.003546
770160,1896641_505604_16-23_2023-11-02 08:14:24.458,1896641,505604,2023-11-02 08:14:24.458,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.001635,-0.000584,0.050650,0.0,33.0,0.0,0.000000,58.0,1.0,0.017241
770161,1896641_25212_16-23_2023-11-02 08:14:25.748,1896641,25212,2023-11-02 08:14:25.748,2023-11-02,0,NaN,NaN,42.0,37.0,...,0.002612,0.000354,0.049871,0.0,33.0,0.0,0.000000,58.0,1.0,0.017241
770162,1900765_95157_16-23_2023-11-02 06:08:25.900,1900765,95157,2023-11-02 06:08:25.900,2023-11-02,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
from sklearn.model_selection import GroupShuffleSplit

# --- Split 1: Small sample train/test (10% of data) for quick experiments ---
unique_ids = train_copy['id2'].unique()
np.random.seed(42)
small_ids = np.random.choice(unique_ids, size=int(len(unique_ids) * 0.1), replace=False)
train_small = train_copy[train_copy['id2'].isin(small_ids)].copy()

gss_small = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_s, test_idx_s = next(gss_small.split(train_small, groups=train_small['id2']))

small_train_df = train_small.iloc[train_idx_s].reset_index(drop=True)
small_test_df = train_small.iloc[test_idx_s].reset_index(drop=True)

print("=== Small Split (10% sample) ===")
print(f"Train: {len(small_train_df)} rows, {small_train_df['id2'].nunique()} groups")
print(f"Test:  {len(small_test_df)} rows, {small_test_df['id2'].nunique()} groups")
assert set(small_train_df['id2']).isdisjoint(set(small_test_df['id2'])), "Group leakage!"

# Save small split
small_train_df.to_parquet(os.path.join(Project_Dir, Data_dir, "small_train.parquet"), engine="pyarrow", index=False)
small_test_df.to_parquet(os.path.join(Project_Dir, Data_dir, "small_test.parquet"), engine="pyarrow", index=False)
print("✓ Saved small_train.parquet and small_test.parquet")

# --- Split 2: Full train/test on entire train_copy ---
gss_full = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_f, test_idx_f = next(gss_full.split(train_copy, groups=train_copy['id2']))

full_train_df = train_copy.iloc[train_idx_f].reset_index(drop=True)
full_test_df = train_copy.iloc[test_idx_f].reset_index(drop=True)

print("\n=== Full Split ===")
print(f"Train: {len(full_train_df)} rows, {full_train_df['id2'].nunique()} groups")
print(f"Test:  {len(full_test_df)} rows, {full_test_df['id2'].nunique()} groups")
assert set(full_train_df['id2']).isdisjoint(set(full_test_df['id2'])), "Group leakage!"

# Save full split
full_train_df.to_parquet(os.path.join(Project_Dir, Data_dir, "full_train.parquet"), engine="pyarrow", index=False)
full_test_df.to_parquet(os.path.join(Project_Dir, Data_dir, "full_test.parquet"), engine="pyarrow", index=False)
print("✓ Saved full_train.parquet and full_test.parquet")

=== Small Split (10% sample) ===
Train: 61076 rows, 3724 groups
Test:  14467 rows, 931 groups
✓ Saved small_train.parquet and small_test.parquet

=== Full Split ===
Train: 616602 rows, 37240 groups
Test:  153544 rows, 9310 groups
✓ Saved full_train.parquet and full_test.parquet


In [36]:
train_copy.groupby('id2').size().describe()

count    46550.000000
mean        16.544490
std         36.753124
min          1.000000
25%          2.000000
50%          3.000000
75%          6.000000
max        510.000000
dtype: float64

In [37]:
# ── Export small datasets to CSV ──────────────────────────────────────────────
csv_dir = os.path.join(Project_Dir, Data_dir, "csv")
os.makedirs(csv_dir, exist_ok=True)

small_train.to_csv(os.path.join(csv_dir, "small_train.csv"), index=False)
small_test.to_csv(os.path.join(csv_dir, "small_test.csv"),  index=False)

print(f"Saved small_train.csv  →  {os.path.join(csv_dir, 'small_train.csv')}")
print(f"Saved small_test.csv   →  {os.path.join(csv_dir, 'small_test.csv')}")

Saved small_train.csv  →  /Users/akashjaiswal/Desktop/amex-hackathon-2/Data/csv/small_train.csv
Saved small_test.csv   →  /Users/akashjaiswal/Desktop/amex-hackathon-2/Data/csv/small_test.csv


In [38]:
full_train_df.to_csv(os.path.join(Project_Dir, Data_dir, "full_train.csv"), index=False)
full_test_df.to_csv(os.path.join(Project_Dir, Data_dir, "full_test.csv"), index=False)
print("✓ Saved full_train.csv and full_test.csv")



✓ Saved full_train.csv and full_test.csv
